# 00 - DOWNLOAD DATASET

In [1]:
import wfdb 
import requests
from bs4 import BeautifulSoup
import re
import os
import pandas as pd

## 1 - Filter Cardiac Surgery ICD Procedures

In [2]:
#Filtar codigos de procedimientos
import pandas as pd
df = pd.read_csv('MIMIC-IV/d_icd_procedures.csv')

include_keywords = [
    "cardiac", "heart", "coronary", "valve", "myocard", "bypass",
    "aortic", "ventricular", "atri", "septal", "mitral", "pulmonary valve",
    "tricuspid", "endocard", "epicard", "pericard"
]
# Palabras clave para excluir (evitar falsos positivos de otros sistemas)
exclude_keywords = [
    "cerebral", "spinal", "nerve", "cranial", "brain",
    "bone marrow", "ear", "eye", "lung", "trachea", "bronch",
    "skin", "larynx", "pharynx", "esophagus", "bladder", "kidney",
    "liver", "pancreas", "stomach", "intestinal", "colon", "rectum"
]
# Máscara para incluir y excluir
mask_include = df['long_title'].str.contains('|'.join(include_keywords), case=False, na=False)
mask_exclude = df['long_title'].str.contains('|'.join(exclude_keywords), case=False, na=False)

# Filtrar solo cirugía cardíaca/vascular
cardiac_vascular_df = df[mask_include & ~mask_exclude]

# Guardar resultados
cardiac_vascular_df.to_csv("aux_dataset/cardiac_surgery_icd_procedures.csv", index=False)

print(f"Se han guardado {len(cardiac_vascular_df)} códigos en aux_dataset/cardiac_surgery_icd_procedures.csv")


Se han guardado 6209 códigos en aux_dataset/cardiac_surgery_icd_procedures.csv


## 2 - Create Cardiac Surgery Patients Dataset

In [3]:
# Create Cardiac Surgery Patients Dataset

# Cargar datos
procedures_df = pd.read_csv('MIMIC-IV/procedures_icd.csv', parse_dates=['chartdate'])
admissions_df = pd.read_csv('MIMIC-IV/admissions.csv', parse_dates=['deathtime'])
cardiac_codes_df = pd.read_csv('aux_dataset/cardiac_surgery_icd_procedures.csv')

# Lista de códigos de cirugía cardíaca
cardiac_codes = cardiac_codes_df['icd_code'].unique()

# Filtrar procedimientos de cirugía cardíaca
cardiac_procedures_df = procedures_df[procedures_df['icd_code'].isin(cardiac_codes)]

# Merge (puede generar subject_id_x y subject_id_y)
merged_df = pd.merge(
    cardiac_procedures_df,
    admissions_df[['subject_id', 'hadm_id', 'deathtime']],
    on='hadm_id',
    how='inner'
)

# Si hay dos columnas de subject_id, unificamos
if 'subject_id_x' in merged_df.columns:
    merged_df = merged_df.rename(columns={'subject_id_x': 'subject_id'})
if 'subject_id_y' in merged_df.columns:
    merged_df['subject_id'] = merged_df['subject_id_y']
    merged_df = merged_df.drop(columns=['subject_id_y'])

# Agrupar por paciente
procedures_per_patient = (
    merged_df.groupby('subject_id')
    .agg(
        procedures=('icd_code', lambda codes: ';'.join(sorted(codes.unique()))),
        procedure_date=('chartdate', 'max'),
        deathtime=('deathtime', 'max')
    )
    .reset_index()
)

# Calcular si murió dentro de los 30 días posteriores a la última intervención
procedures_per_patient['mortality_30d'] = (
    (procedures_per_patient['deathtime'].notna()) &
    ((procedures_per_patient['deathtime'] - procedures_per_patient['procedure_date']).dt.days <= 30)
)

# Guardar resultado
procedures_per_patient[['subject_id', 'procedures', 'procedure_date', 'deathtime', 'mortality_30d']].to_csv(
    'aux_dataset/30dm_cardiac_surgery_dataset.csv',
    index=False
)
print("Dataset de pacientes de cirugía cardíaca guardado en aux_dataset/30dm_cardiac_surgery_dataset.csv")


Dataset de pacientes de cirugía cardíaca guardado en aux_dataset/30dm_cardiac_surgery_dataset.csv


In [20]:
print(f'Initial MIMIC-IV population: {admissions_df["subject_id"].nunique()}')
print(f'Cardiac surgery patiens (filtered by IDC): {procedures_per_patient["subject_id"].nunique()}')

Initial MIMIC-IV population: 223452
Cardiac surgery patiens (filtered by IDC): 34385


## 3 - Create ECG Cardiac Surgery Dataset

In [3]:
# Create ECG Cardiac Surgery Dataset
cardiac_surgery_df = pd.read_csv('aux_dataset/30dm_cardiac_surgery_dataset.csv')
ecg_index_df = pd.read_csv('MIMIC-IV/ECG/record_list.csv')

#Crear un nuevo df que sea ecg_index:df, pero unicamente los subject_id que coincidan con los de cardiac_surgery_df y si hay varios reghistros subject_id con el mismo npmbre, cpger el que en ecg_time sea más nuevo

ecg_cardiac_surgery_df = ecg_index_df[ecg_index_df['subject_id'].isin(cardiac_surgery_df['subject_id'])]
ecg_cardiac_surgery_df = ecg_cardiac_surgery_df.loc[ecg_cardiac_surgery_df.groupby('subject_id')['ecg_time'].idxmax()]
#Añade tambien de cardiac_surgery los procedures, el prodecure date el deathtime y el mortality_30d

ecg_cardiac_surgery_df = ecg_cardiac_surgery_df.merge(cardiac_surgery_df[['subject_id', 'procedures', 'procedure_date', 'deathtime', 'mortality_30d']], on='subject_id', how='left')

ecg_cardiac_surgery_df.to_csv('aux_dataset/ecg_cardiac_surgery_dataset.csv', index=False)
print("Dataset de ECG de pacientes de cirugía cardíaca guardado en aux_dataset/ecg_cardiac_surgery_dataset.csv")

Dataset de ECG de pacientes de cirugía cardíaca guardado en aux_dataset/ecg_cardiac_surgery_dataset.csv


## 4 - Download ECG Records

In [4]:
# Download ECG Records
import csv

download_status = set()
log_file = 'LOGS/ecg_downloading_log.csv'

# Inicializa el archivo de log si no existe
if not os.path.exists(log_file):
    with open(log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'filename', 'extension', 'status', 'http_code', 'error'])

for id in range(ecg_cardiac_surgery_df.shape[0]):
    storage_path = f'/Volumes/KINGSTON/MIMIC-IV/ECG/{ecg_cardiac_surgery_df.iloc[id]["path"]}'
    storage_path = storage_path[:storage_path.rfind('/')]
    download_path = f'https://physionet.org/files/mimic-iv-ecg/1.0/{ecg_cardiac_surgery_df.iloc[id]["path"]}'
    filename = ecg_cardiac_surgery_df.iloc[id]["path"]
    if not os.path.exists(storage_path):
        os.makedirs(storage_path, exist_ok=True)
        print(f"Descargando {storage_path}...")
        

        file_extension = [".dat", ".hea"]
        for ext in file_extension:
            
            download_url = f'{download_path}{ext}?download'
            print(f"Desde {download_url}...")
            status = 'error'
            http_code = ''
            error_msg = ''
            try:
                response = requests.get(download_url, timeout=30)
                http_code = response.status_code
                if response.status_code == 200:
                    lines = response.text.strip().split('\n')
                    for line in lines:
                        download_status.add(line.strip())
                    local_file = f'{download_path}{ext}'.replace('https://physionet.org/files/mimic-iv-ecg/1.0/', 'MIMIC-IV/ECG/')
                    with open(local_file, 'w') as f:
                        f.write('\n'.join(lines))
                    status = 'descargado'
                else:
                    error_msg = f'HTTP {response.status_code}'
                    print(f"Error al descargar {storage_path}: {response.status_code}")
            except Exception as e:
                error_msg = str(e)
                print(f"Excepción al descargar {storage_path}: {e}")
            # Registrar en el log
            with open(log_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([id, filename, ext, status, http_code, error_msg])


Descargando /Volumes/KINGSTON/MIMIC-IV/ECG/files/p1111/p11115360/s49988961...
Desde https://physionet.org/files/mimic-iv-ecg/1.0/files/p1111/p11115360/s49988961/49988961.dat?download...
Desde https://physionet.org/files/mimic-iv-ecg/1.0/files/p1111/p11115360/s49988961/49988961.hea?download...


KeyboardInterrupt: 

In [ ]:
import os
import requests
import pandas as pd
import csv

# URL base
base_url = "https://physionet.org/files/mimic-iv-ecg/1.0/"

# Leer CSV con paths
df = pd.read_csv("MIMIC-IV/ECG/record_list.csv")

# Carpeta raíz donde guardar
root_folder = '/Volumes/KINGSTON/MIMIC-IV/ECG/'

# Archivo de log
log_file = 'LOGS/ecg_downloading_log.csv'
os.makedirs(os.path.dirname(log_file), exist_ok=True)

# Inicializa el archivo de log si no existe
if not os.path.exists(log_file):
    with open(log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'filename', 'extension', 'status', 'http_code', 'error'])

# Descarga de archivos
for idx, row in df.iterrows():
    local_folder = os.path.join(root_folder, os.path.dirname(row['path']))
    os.makedirs(local_folder, exist_ok=True)

    for ext in ['.dat', '.hea']:
        download_url = f"{base_url}{row['path']}{ext}"
        local_file = os.path.join(local_folder, os.path.basename(row['path']) + ext)

        status = 'error'
        http_code = ''
        error_msg = ''

        if os.path.exists(local_file):
            status = 'ya existente'
            http_code = 200
        else:
            try:
                print(f"Descargando {download_url} ...")
                response = requests.get(download_url, timeout=30)
                http_code = response.status_code
                if response.status_code == 200:
                    with open(local_file, 'wb') as f:
                        f.write(response.content)
                    status = 'descargado'
                else:
                    error_msg = f"HTTP {response.status_code}"
                    print(f"Error al descargar {download_url}: {response.status_code}")
            except Exception as e:
                error_msg = str(e)
                print(f"Excepción al descargar {download_url}: {e}")

        # Guardar en log
        with open(log_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([idx, row['path'], ext, status, http_code, error_msg])
